In [ ]:
import pathlib

import numpy as np
import pandas as pd
import cv2 as cv
import mediapipe as mp

from src.letter_detection.points_normalization import normalize_lists

In [ ]:
PATH_TO_VIDEOS = 'C:/Users/domur/Videos/ASL_dataset_videos'
videos_path = pathlib.Path(PATH_TO_VIDEOS)

In [ ]:
for video in videos_path.iterdir():
    print(video)

In [ ]:
mpHands = mp.solutions.hands
hands = mpHands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)
mpDraw = mp.solutions.drawing_utils

In [ ]:
def draw_circles(xs, ys, radius=10, img_size=480, white=False):
    if white:
        img = np.ones((img_size, img_size, 3), dtype=np.uint8) * 255
    else:
        img = np.zeros((img_size, img_size, 3), dtype=np.uint8)

    for i, (x, y) in enumerate(zip(xs, ys)):
        x = int(x * img_size)
        y = int(y * img_size)
        color_code = i % 4
        match color_code:
            case 0:
                color = (0, 0, 255)
            case 1:
                color = (0, 255, 0)
            case 2:
                color = (255, 0, 0)
            case 3:
                color = (255, 255, 0)
            case _:
                color = (0, 0, 0)
        cv.circle(img, (x, y), radius, color, -1)

    return img


import math


def rotate_points(points):
    origin = points[0]
    rotated_points_x = [origin[0]]
    rotated_points_y = [origin[1]]

    p0, p9 = points[0], points[9]
    dx, dy = p9[0] - p0[0], p9[1] - p0[1]
    alpha = math.atan(dx / dy)

    if alpha > 3.14 / 4:
        alpha -= 3.14 / 2
    elif alpha < -3.14 / 4:
        alpha += 3.14 / 2

    for x, y in points[1:]:
        dx, dy = x - origin[0], y - origin[1]

        new_x = origin[0] + (dx * math.cos(alpha) - dy * math.sin(alpha))
        new_y = origin[1] + (dx * math.sin(alpha) + dy * math.cos(alpha))

        rotated_points_x.append(new_x)
        rotated_points_y.append(new_y)

    return rotated_points_x, rotated_points_y


def normalize_lists(xs, ys, rotate=True):
    if rotate:
        xs, ys = rotate_points(list(zip(xs, ys)))
    min_x = min(xs)
    max_x = max(xs)
    min_y = min(ys)
    max_y = max(ys)

    diff_x = max_x - min_x
    diff_y = max_y - min_y

    scale = max(diff_x, diff_y)

    normalized_xs = [(x - min_x) / scale for x in xs]
    normalized_ys = [(y - min_y) / scale for y in ys]

    return normalized_xs, normalized_ys


In [ ]:
CSV_FILE_NAME = 'video_dataset_fhpq_no_rotate.csv'
# Generate x and y columns
x_columns = [f"x{i:02}" for i in range(21)]
y_columns = [f"y{i:02}" for i in range(21)]
z_columns = [f"z{i:02}" for i in range(21)]

# Generate the letters excluding 'J' and 'Z'
letters = [chr(i) for i in range(65, 91) if chr(i) not in ['J', 'Z']]

columns = x_columns + y_columns + z_columns + letters
# Combine columns
header = pd.DataFrame(columns=columns)
header.to_csv(CSV_FILE_NAME, mode='a', index=False)
header

In [ ]:
break_flag = False
current_letter = None
no_rotate_letters = {'F', 'H', 'P', 'Q'}
for video in videos_path.iterdir():
    current_letter = video.name.split('.')[0]
    cnt = 0
    if break_flag:
        break
    cap = cv.VideoCapture(str(video))
    while cap.isOpened():
        success, img = cap.read()
        if not success or cnt > 800:
            break
        imgRGB = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        results = hands.process(imgRGB)

        if results.multi_hand_landmarks:
            cnt += 1
            handLms = results.multi_hand_landmarks[0]
            mpDraw.draw_landmarks(img, handLms, mpHands.HAND_CONNECTIONS)
            xs = [p.x for p in handLms.landmark]
            ys = [p.y for p in handLms.landmark]
            zs = [p.z for p in handLms.landmark]
            if current_letter not in no_rotate_letters:
                list_normalized_xs, list_normalized_ys = normalize_lists(xs, ys)
            else:
                list_normalized_xs, list_normalized_ys = normalize_lists(xs, ys, rotate=False)
            df_row = pd.DataFrame([
                list_normalized_xs + list_normalized_ys + zs + [int(current_letter == letter) for letter in letters]])
            df_row.to_csv(CSV_FILE_NAME, mode='a', index=False, header=False)
        print(current_letter, cnt)
        if cv.waitKey(1) & 0xFF == ord('q'):
            break_flag = True
            break
        if cv.waitKey(1) & 0xFF == ord('n'):
            break
    print(current_letter, cnt)
    cap.release()
    cv.destroyAllWindows()